# Notebook 02 — Preprocessing Data

**Tujuan:** Membersihkan data, menghitung fitur proporsi, dan menormalisasi dengan Z-Score (StandardScaler) sebelum clustering.

Tahapan:
1. Load data mentah via `load_dtsen()`
2. Hitung 8 fitur proporsi via `buat_fitur_proporsi()`
3. Validasi kualitas data (NaN, range)
4. Visualisasi distribusi sebelum normalisasi
5. Normalisasi StandardScaler
6. Visualisasi distribusi setelah normalisasi
7. Simpan `dtsen_clean.csv`, `dtsen_scaled.csv`, `scaler.pkl`

In [1]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import (
    load_dtsen, buat_fitur_proporsi, FEATURE_COLS,
    DATA_DIR, FIGURES_DIR, RESULTS_DIR
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10})

print('Library berhasil diimport.')
print(f'RESULTS: {RESULTS_DIR.resolve()}')

Library berhasil diimport.
RESULTS: analysis\output\results


In [2]:
# ── Cell 3: Load Data ──────────────────────────────────────────────────────────
FILE_EXCEL = DATA_DIR / '20_JULI_2026-_Rekap_DTSEN_Kabupaten_Lahat.xls'

df = load_dtsen(FILE_EXCEL)

print(f'Shape setelah load & cleaning : {df.shape}')
print(f'Jumlah kecamatan               : {df["KECAMATAN"].nunique()}')
print(f'Jumlah desa/kelurahan          : {len(df)}')
print(f'NaN per kolom kunci:')
print(df[['JUMLAH_KELUARGA','DESIL1_KLG','DESIL2_KLG','DESIL3_KLG','DESIL4_KLG',
          'DESIL5_KLG','DESIL6_10_KLG','BELUM_PRMK_KLG','NONAKTIF_KLG']].isna().sum())

ℹ  Desa tanpa profil desil valid (di-exclude):
KECAMATAN   KELURAHAN  JUMLAH_KELUARGA
    Lahat Talang Jawa                1

  DTSEN Kabupaten Lahat — Data Loaded
  Baris (desa/kel)  : 371
  Kolom             : 20
  Kecamatan         : 24
  Total KK          : 142,799
  Total Individu    : 450,323

Shape setelah load & cleaning : (371, 20)
Jumlah kecamatan               : 24
Jumlah desa/kelurahan          : 371
NaN per kolom kunci:
JUMLAH_KELUARGA    0
DESIL1_KLG         0
DESIL2_KLG         0
DESIL3_KLG         0
DESIL4_KLG         0
DESIL5_KLG         0
DESIL6_10_KLG      0
BELUM_PRMK_KLG     0
NONAKTIF_KLG       0
dtype: int64


In [3]:
# ── Cell 4: Hitung Fitur Proporsi & Validasi ───────────────────────────────────
df_feat = buat_fitur_proporsi(df)

print(f'Shape df_feat  : {df_feat.shape}')
print(f'Kolom          : {list(df_feat.columns)}')

# Validasi NaN
nan_count = df_feat[FEATURE_COLS].isna().sum()
print(f'\nJumlah NaN per fitur:')
print(nan_count)
assert nan_count.sum() == 0, 'Terdapat NaN pada fitur proporsi — periksa data sumber!'

# Validasi range (setiap nilai harus 0–100)
for col in FEATURE_COLS:
    mn, mx = df_feat[col].min(), df_feat[col].max()
    if mn < -0.01 or mx > 100.01:
        print(f'  PERINGATAN: {col} di luar range [0,100] → min={mn:.4f}, max={mx:.4f}')
    else:
        print(f'  OK {col:20s}: min={mn:6.2f}%, max={mx:6.2f}%')

print('\n✅ Validasi selesai — tidak ada NaN, semua nilai dalam range 0–100%')

Shape df_feat  : (371, 11)
Kolom          : ['KECAMATAN', 'KELURAHAN', 'JUMLAH_KELUARGA', 'pct_desil1', 'pct_desil2', 'pct_desil3', 'pct_desil4', 'pct_desil5', 'pct_desil6_10', 'pct_belum_prmk', 'pct_nonaktif']

Jumlah NaN per fitur:
pct_desil1        0
pct_desil2        0
pct_desil3        0
pct_desil4        0
pct_desil5        0
pct_desil6_10     0
pct_belum_prmk    0
pct_nonaktif      0
dtype: int64
  OK pct_desil1          : min=  2.94%, max= 44.37%
  OK pct_desil2          : min=  2.75%, max= 35.78%
  OK pct_desil3          : min=  1.69%, max= 21.65%
  OK pct_desil4          : min=  1.59%, max= 17.54%
  OK pct_desil5          : min=  2.58%, max= 29.41%
  OK pct_desil6_10       : min= 12.84%, max= 74.52%
  OK pct_belum_prmk      : min=  0.00%, max= 12.83%
  OK pct_nonaktif        : min=  2.00%, max= 16.92%

✅ Validasi selesai — tidak ada NaN, semua nilai dalam range 0–100%


In [4]:
# ── Cell 5: Distribusi Sebelum Normalisasi ─────────────────────────────────────
label_map = {
    'pct_desil1':     'Desil 1 (%)',
    'pct_desil2':     'Desil 2 (%)',
    'pct_desil3':     'Desil 3 (%)',
    'pct_desil4':     'Desil 4 (%)',
    'pct_desil5':     'Desil 5 (%)',
    'pct_desil6_10':  'Desil 6–10 (%)',
    'pct_belum_prmk': 'Belum PRMK (%)',
    'pct_nonaktif':   'Nonaktif KLG (%)',
}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    data = df_feat[col].dropna()
    ax.hist(data, bins=25, color='#4C9BE8', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color='#E84C4C', linestyle='--', linewidth=1.2, label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='#2CA02C', linestyle=':',  linewidth=1.2, label=f'Median={data.median():.1f}')
    ax.set_title(label_map[col], fontsize=9)
    ax.set_xlabel('%', fontsize=8)
    ax.legend(fontsize=6)

fig.suptitle('Distribusi 8 Fitur Proporsi — SEBELUM Normalisasi', fontweight='bold', y=1.01)
plt.tight_layout()
outpath = FIGURES_DIR / '02_distribusi_sebelum_normalisasi.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Gambar disimpan: analysis\output\figures\02_distribusi_sebelum_normalisasi.png


In [5]:
# ── Cell 6: Normalisasi StandardScaler ────────────────────────────────────────
X = df_feat[FEATURE_COLS].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Statistik setelah normalisasi (harus ≈ mean=0, std=1):')
for i, col in enumerate(FEATURE_COLS):
    print(f'  {col:20s}  mean={X_scaled[:,i].mean():+.4f}  std={X_scaled[:,i].std():.4f}')

# Simpan scaler
scaler_path = RESULTS_DIR / 'scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'\nScaler disimpan: {scaler_path}')

Statistik setelah normalisasi (harus ≈ mean=0, std=1):
  pct_desil1            mean=+0.0000  std=1.0000
  pct_desil2            mean=-0.0000  std=1.0000
  pct_desil3            mean=-0.0000  std=1.0000
  pct_desil4            mean=+0.0000  std=1.0000
  pct_desil5            mean=-0.0000  std=1.0000
  pct_desil6_10         mean=+0.0000  std=1.0000
  pct_belum_prmk        mean=-0.0000  std=1.0000
  pct_nonaktif          mean=+0.0000  std=1.0000

Scaler disimpan: analysis\output\results\scaler.pkl


In [6]:
# ── Cell 7: Distribusi Setelah Normalisasi ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    data = X_scaled[:, i]
    ax.hist(data, bins=25, color='#2CA02C', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color='#E84C4C', linestyle='--', linewidth=1.2, label=f'Mean={data.mean():.2f}')
    ax.axvline(data.std(),    color='#FF7F0E', linestyle=':',  linewidth=1.2, label=f'Std={data.std():.2f}')
    ax.set_title(label_map[col], fontsize=9)
    ax.set_xlabel('Z-Score', fontsize=8)
    ax.legend(fontsize=6)

fig.suptitle('Distribusi 8 Fitur Proporsi — SETELAH Normalisasi Z-Score', fontweight='bold', y=1.01)
plt.tight_layout()
outpath = FIGURES_DIR / '02_distribusi_setelah_normalisasi.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Gambar disimpan: analysis\output\figures\02_distribusi_setelah_normalisasi.png


In [7]:
# ── Cell 8: Simpan Output ─────────────────────────────────────────────────────

# 1. dtsen_clean.csv — fitur proporsi + identitas
clean_path = RESULTS_DIR / 'dtsen_clean.csv'
df_feat.to_csv(clean_path, index=False)
print(f'Disimpan: {clean_path}  shape={df_feat.shape}')

# 2. dtsen_scaled.csv — X_scaled dengan index KELURAHAN
df_scaled = pd.DataFrame(
    X_scaled,
    columns=[f'{c}_z' for c in FEATURE_COLS],
    index=df_feat['KELURAHAN']
)
# Tambahkan kolom identitas
df_scaled.insert(0, 'KECAMATAN', df_feat['KECAMATAN'].values)
df_scaled.insert(1, 'JUMLAH_KELUARGA', df_feat['JUMLAH_KELUARGA'].values)

scaled_path = RESULTS_DIR / 'dtsen_scaled.csv'
df_scaled.to_csv(scaled_path)
print(f'Disimpan: {scaled_path}  shape={df_scaled.shape}')

print(f'\n{"="*50}')
print('  RINGKASAN OUTPUT PREPROCESSING')
print(f'{"="*50}')
print(f'  dtsen_clean.csv  : {df_feat.shape[0]} baris × {df_feat.shape[1]} kolom')
print(f'  dtsen_scaled.csv : {df_scaled.shape[0]} baris × {df_scaled.shape[1]} kolom')
print(f'  scaler.pkl       : StandardScaler tersimpan')
print(f'{"="*50}')
print('\n✅ Preprocessing selesai — siap untuk clustering!')

Disimpan: analysis\output\results\dtsen_clean.csv  shape=(371, 11)
Disimpan: analysis\output\results\dtsen_scaled.csv  shape=(371, 10)

  RINGKASAN OUTPUT PREPROCESSING
  dtsen_clean.csv  : 371 baris × 11 kolom
  dtsen_scaled.csv : 371 baris × 10 kolom
  scaler.pkl       : StandardScaler tersimpan

✅ Preprocessing selesai — siap untuk clustering!


## Ringkasan Preprocessing

| Tahap | Keterangan | Hasil |
|-------|------------|-------|
| Load data | Baca Excel, skip 2 header row, assign 20 kolom | 372 baris mentah |
| Filter subtotal | Hapus baris kecamatan & kabupaten | — |
| Exclude KK=0 | Hapus Talang Jawa (KK=0) | **371 baris** |
| Fitur proporsi | Hitung 8 pct_ dari JUMLAH_KELUARGA | 8 fitur baru |
| Validasi | Cek NaN & range 0–100% | Semua valid |
| Normalisasi | StandardScaler (Z-Score) | mean≈0, std≈1 |

**Output yang dihasilkan:**
- `output/results/dtsen_clean.csv` — 371 × 11 (identitas + 8 fitur)
- `output/results/dtsen_scaled.csv` — 371 × 10 (identitas + 8 fitur Z-scaled)
- `output/results/scaler.pkl` — object StandardScaler untuk transformasi konsisten

> **Langkah selanjutnya:** K-Means Clustering (NB03)